In [ ]:
import pickle
import numpy as np
pkl_path = "/playpen-shared/kechengli/workspace/Fusemoe/FuseMoE/dataset_impute/mimiciv/output_imputed_data/tmp_imputed_64.pkl"

with open(pkl_path, "rb") as f:
    data = pickle.load(f)

# 有些 pkl 是 {"samples": [...]}
if isinstance(data, dict) and "samples" in data:
    samples = data["samples"]
else:
    samples = data

print(f"Total samples: {len(samples)}")

np.set_printoptions(
    precision=4,
    suppress=True,
    linewidth=200
)

for i, s in enumerate(samples[:3]):
    print(f"\n================= 🩺 Sample {i+1} =================")
    print("irg_ts (original):")
    print(np.asarray(s["irg_ts"]))

    print("\nirg_ts_mask:")
    print(np.asarray(s["irg_ts_mask"]))

    print("\nirg_ts_imputed:")
    print(np.asarray(s["irg_ts_imputed"]))

    if "moe_w_text_cxr_ecg" in s:
        print("\nMoE weights [text, cxr, ecg]:")
        print(s["moe_w_text_cxr_ecg"])





================= 🩺 Sample 1 =================
name: 19106955
hadm_id: 27698713
stay_id: 38191620
MoE weights [text, cxr, ecg]: [9.7166926e-01 4.2176299e-04 2.7909040e-02]

Time series shape: L=94, K=30

--- t = 0 ---
t=000, k=00 | orig=    0.0000 | imputed=   -0.0574 | MISS
t=000, k=01 | orig=    0.0000 | imputed=   -0.5014 | MISS
t=000, k=02 | orig=    0.0000 | imputed=    0.9397 | MISS
t=000, k=03 | orig=    0.0000 | imputed=   -0.0023 | MISS
t=000, k=04 | orig=   -0.0095 | imputed=   -0.0095 | OBS
t=000, k=05 | orig=   -0.0160 | imputed=   -0.0160 | OBS
t=000, k=06 | orig=    0.0000 | imputed=   -0.0343 | MISS
t=000, k=07 | orig=    0.0000 | imputed=   -0.0477 | MISS
t=000, k=08 | orig=    0.0000 | imputed=   -0.1162 | MISS
t=000, k=09 | orig=    0.0000 | imputed=    0.4244 | MISS
t=000, k=10 | orig=    0.0000 | imputed=   -1.1176 | MISS
t=000, k=11 | orig=   -0.0205 | imputed=   -0.0205 | OBS
t=000, k=12 | orig=    0.0000 | imputed=   -0.0698 | MISS
t=000, k=13 | orig=   -0.0157 

In [2]:
for i, s in enumerate(samples[:3]):
    print(f"\n🔍 Sanity check Sample {i+1}")
    
    if not all(k in s for k in ["irg_ts", "irg_ts_imputed", "irg_ts_mask"]):
        print("missing required keys, skip")
        continue

    x = np.asarray(s["irg_ts"])
    x_imp = np.asarray(s["irg_ts_imputed"])
    mask = np.asarray(s["irg_ts_mask"]).astype(bool)

    if mask.any():
        max_diff = np.abs(x[mask] - x_imp[mask]).max()
        print("max diff on observed points:", max_diff)
    else:
        print("no observed points in this sample")



🔍 Sanity check Sample 1
max diff on observed points: 8.961711106536541e-08

🔍 Sanity check Sample 2
max diff on observed points: 1.1794343324211809e-07

🔍 Sanity check Sample 3
max diff on observed points: 1.0882195899952762e-07


In [3]:
# 只看第一个样本
s = samples[0]

x = np.asarray(s["irg_ts"])
x_imp = np.asarray(s["irg_ts_imputed"])
mask = np.asarray(s["irg_ts_mask"]).astype(bool)

# 找几个缺失点
missing_idx = np.argwhere(~mask)

print("Number of missing points:", len(missing_idx))

for idx in missing_idx[:5]:
    t, k = idx
    print(f"(t={t}, k={k}) original={x[t,k]}  imputed={x_imp[t,k]}")


Number of missing points: 2382
(t=0, k=0) original=0.0  imputed=-0.057365309447050095
(t=0, k=1) original=0.0  imputed=-0.5014367699623108
(t=0, k=2) original=0.0  imputed=0.9397284388542175
(t=0, k=3) original=0.0  imputed=-0.0022990216966718435
(t=0, k=6) original=0.0  imputed=-0.034255076199769974
